<center><img src='https://drive.google.com/uc?id=1_utx_ZGclmCwNttSe40kYA6VHzNocdET' height="60"></center>

AI TECH - Akademia Innowacyjnych Zastosowań Technologii Cyfrowych. Program Operacyjny Polska Cyfrowa na lata 2014-2020
<hr>

<center><img src='https://drive.google.com/uc?id=1BXZ0u3562N_MqCLcekI-Ens77Kk4LpPm'></center>

<center>
Projekt współfinansowany ze środków Unii Europejskiej w ramach Europejskiego Funduszu Rozwoju Regionalnego
Program Operacyjny Polska Cyfrowa na lata 2014-2020,
Oś Priorytetowa nr 3 "Cyfrowe kompetencje społeczeństwa" Działanie  nr 3.2 "Innowacyjne rozwiązania na rzecz aktywizacji cyfrowej"
Tytuł projektu:  „Akademia Innowacyjnych Zastosowań Technologii Cyfrowych (AI Tech)”
    </center>

## Laboratory Scenario 2 - Backpropagation and Gradient Checkpointing

In this lab scenario, you are given an implementation of a simple neural network, and your goal is to implement the backpropagation procedure for this network.  

To be more precise, the network inputs a tensor $x$ of shape `(MINI_BATCH_SIZE, 28*28)`, where each element of the batch represents a flattened grayscale image of shape `(28, 28)`.  
In exercise 1, you can assume that images in the minibatch are fed to the network one by one (as tensors of shape `(1, 28*28)` - single image and `(1, 10)` - image class).  
In exercise 2 you are asked to make the backpropagation work without this assumption, on whole mini-batches.  
In exercise 3, you will implement a technique called *gradient checkpointing*, that allows you to reduce the amount of memory used to store activations for backpropagation.

In [ ]:
from pathlib import Path
from typing import Sequence, Iterator, cast
from typing_extensions import override

import numpy as np
import PIL.Image
from numpy.typing import NDArray
from tqdm.auto import tqdm

FloatNDArray = NDArray[np.float64]

## Loading the MNIST dataset

In [ ]:
!wget --no-verbose -O mnist.npz https://s3.amazonaws.com/img-datasets/mnist.npz

In [ ]:
def load_mnist(
    path: Path = Path("mnist.npz"),
) -> tuple[FloatNDArray, FloatNDArray, FloatNDArray, FloatNDArray]:
    """
    Load the MNIST dataset (grayscale 28 x 28 images of hand-written digits).

    Returns tuple of:
    - x_train: shape (N_train, H * W), grayscale values 0..1.
    - y_train: shape (N_train, 10), one-hot-encoded label, dtype float64.
    - x_test: shape (N_test, H * W), grayscale values 0..1.
    - y_train: shape (N_test, 10), one-hot-encoded label, dtype float64.

    More: https://en.wikipedia.org/wiki/MNIST_database
    """
    with np.load(path) as f:
        x_train, _y_train = f["x_train"], f["y_train"]
        x_test, _y_test = f["x_test"], f["y_test"]

    H = W = 28
    N_train = len(x_train)
    N_test = len(x_test)
    assert x_train.shape == (N_train, H, W) and _y_train.shape == (N_train,)
    assert x_test.shape == (N_test, H, W) and _y_test.shape == (N_test,)

    x_train = x_train.reshape(N_train, H * W) / 255.0
    x_test = x_test.reshape(N_test, H * W) / 255.0

    # One-hot encode the labels.
    y_train = np.zeros((N_train, 10), dtype=np.float64)
    y_train[np.arange(N_train), _y_train] = 1

    y_test = np.zeros((N_test, 10))
    y_test[np.arange(N_test), _y_test] = 1

    return x_train, y_train, x_test, y_test


x_train, y_train, x_test, y_test = load_mnist()

In [ ]:
def to_pillow_image(x: FloatNDArray, scale: int = 10) -> PIL.Image.Image:
    """Convert example of shape (28 * 28,) and values 0..1 to Pillow image."""
    H, W = 28, 28
    assert x.shape == (H * W,) and 0 <= x.min() <= x.max() <= 1
    x_int = (x * 255).astype(np.uint8)
    img = PIL.Image.fromarray(x_int.reshape(H, W))
    return img.resize((H * scale, W * scale), PIL.Image.Resampling.NEAREST)


display(to_pillow_image(x_train[0]))
print(y_train[0])

## Network and trainer

Here is a basic implementation of an MLP (Multi-Layer Perceptron, a simple neural network with fully-connected layers and sigmoid activation) and SGD (Stochastic Gradient Descent) for it.<br>
In Exercise 1 you will complete it by implementing the `backprop_single` method.<br>
In Exercise 2 you will then override `feedforward` and `backprop` to make vectorized versions (working on whole mini-batches with vectorized operations).

In [ ]:
def sigmoid(z: FloatNDArray) -> FloatNDArray:
    return 1.0 / (1.0 + np.exp(-z))

#### BaseNetwork

In [ ]:
class BaseNetwork:
    """Multi-Layer Perceptron"""

    def __init__(self, sizes: Sequence[int] = (784, 30, 10)) -> None:
        """
        Args:
        - sizes: sequence of layer widths [N^0, ... , N^last]
          These are lengths of activation vectors, where:
          - N^0 is input size: 784.
          - N^last is the number of classes into which we can classify each input: 10.
        """
        self.sizes = list(sizes)

        # We initialize weights and biases from a random normal distribution.

        # List of len(sizes) - 1 vectors of shape (N^1), (N^2), ..., (N^last).
        self.biases = [np.random.randn(n) for n in sizes[1:]]

        # List of len(sizes) - 1 matrices of shape (N^{i+1}, N^i}).
        # Weights are indexed by target node first.
        self.weights = [
            np.random.randn(n_out, n_in) / np.sqrt(n_in)
            for n_in, n_out in zip(sizes[:-1], sizes[1:], strict=True)
        ]

    def feedforward_single(self, x: FloatNDArray) -> FloatNDArray:
        """
        Run the network on a single case of shape (N^0,).
        Returns last layer activations, shape (N^last,).
        """
        g = x
        for w, b in zip(self.weights, self.biases, strict=True):
            f = w @ g + b  # pre-activations
            g = sigmoid(f)  # activations
        return g

    def feedforward(self, x_mini_batch: FloatNDArray) -> FloatNDArray:
        """
        Run the network on a batch of cases of shape (B, N^0).
        Returns last layer activations, shape (B, N^last).
        """
        B = len(x_mini_batch)
        return np.array([self.feedforward_single(x_mini_batch[i]) for i in range(B)])

    def backprop_single(
        self, x: FloatNDArray, y: FloatNDArray
    ) -> tuple[list[FloatNDArray], list[FloatNDArray]]:
        """
        Backpropagation for a single input image.

        Args:
        - x: input features, shape (N^0,).
        - y: target label (one-hot encoded), shape (N^last,).

        Returns (grads_w, grads_b), where:
        - grads_w is a list of gradients over weights for each layer, shapes (N^{i+1}, N^i).
        - grads_b is a list of gradients over biases for each layer, shapes (N^i).
        """
        raise NotImplementedError()

    def backprop(
        self, x_mini_batch: FloatNDArray, y_mini_batch: FloatNDArray
    ) -> tuple[list[FloatNDArray], list[FloatNDArray]]:
        """
        Backpropagation for a mini-batch.

        Args:
        - x_mini_batch: input, shape (B, N^0)
        - y_mini_batch: target label (one-hot encoded), shape (B, N^last)

        Returns (grads_w, grads_b), where:
        - grads_w is a list of gradients over weights for each layer, shapes (N^{i+1}, N^i).
        - grads_b is a list of gradients over biases for each layer, shapes (N^i).
        """
        # Accumulate gradients by running backprop one dataitem at a time (no vectorization).
        # Here we define the mini-batch loss as the _mean_ loss over dataitems,
        # so the mini-batch gradient is also the mean of the individual gradients: sum / B.
        B = len(x_mini_batch)
        grads_w = [np.zeros(w.shape) for w in self.weights]
        grads_b = [np.zeros(b.shape) for b in self.biases]
        for x, y in zip(x_mini_batch, y_mini_batch):
            item_grads_w, item_grads_b = self.backprop_single(x, y)
            for i in range(len(grads_w)):
                grads_w[i] = grads_w[i] + item_grads_w[i] / B
            for i in range(len(grads_b)):
                grads_b[i] = grads_b[i] + item_grads_b[i] / B
        return grads_w, grads_b

#### Trainer

In [ ]:
class Trainer:
    """Stochastic Gradient Descent (SGD) optimizer for a BaseNetwork."""

    def __init__(self, network: BaseNetwork) -> None:
        self.network = network

    def evaluate(
        self,
        x_test_data: FloatNDArray,
        y_test_data: FloatNDArray,
        mini_batch_size: int = 1024,
    ) -> float:
        """
        Compute accuracy: the ratio of correct answers for test_data.

        Args (here M is the number of test dataitems):
        - x_test_data: shape (M, N^0).
        - y_test_data: shape (M, N^last).
        - mini_batch_size.
        """
        M = len(x_test_data)
        B = mini_batch_size
        num_mini_batches = (M + B - 1) // B  # ceil(M / B)

        n_correct = 0
        for i in range(num_mini_batches):
            i_begin, i_end = i * B, min((i + 1) * B, M)
            x_mini_batch = x_test_data[i_begin:i_end]
            y_mini_batch = y_test_data[i_begin:i_end]

            predicted_probabilities = self.network.feedforward(
                x_mini_batch
            )  # shape (B, N^last)
            predictions = np.argmax(predicted_probabilities, axis=1)  # shape (B,)
            targets = np.argmax(y_mini_batch, axis=1)  # shape (B,)
            n_correct += np.sum(predictions == targets)

        return n_correct / M if M else 0.0

    def learning_step(
        self,
        x_mini_batch: FloatNDArray,
        y_mini_batch: FloatNDArray,
        learning_rate: float,
    ) -> None:
        """
        Update network parameters with a single mini-batch step of backpropagation and gradient descent.

        Args:
        - x_mini_batch: shape (B, N^0) where B is mini_batch_size.
        - y_mini_batch: shape (B, N^last).
        - learning_rate.
        """
        grads_w, grads_b = self.network.backprop(x_mini_batch, y_mini_batch)

        # Gradient descent step.
        self.network.weights = [
            w - learning_rate * grad_w
            for w, grad_w in zip(self.network.weights, grads_w, strict=True)
        ]
        self.network.biases = [
            b - learning_rate * grad_b
            for b, grad_b in zip(self.network.biases, grads_b, strict=True)
        ]

    def train(
        self,
        training_data: tuple[FloatNDArray, FloatNDArray],
        test_data: tuple[FloatNDArray, FloatNDArray] | None = None,
        epochs: int = 2,
        mini_batch_size: int = 100,
        learning_rate: float = 1.0,
    ) -> None:
        x_train, y_train = training_data

        for epoch in tqdm(range(epochs)):
            # Train epoch.
            M = len(x_train)
            B = mini_batch_size
            num_mini_batches = (M + B - 1) // B  # ceil(M / B)

            for i in range(num_mini_batches):
                i_begin, i_end = i * B, min((i + 1) * B, M)
                x_mini_batch = x_train[i_begin:i_end]
                y_mini_batch = y_train[i_begin:i_end]
                self.learning_step(x_mini_batch, y_mini_batch, learning_rate)

            # Evaluate after each epoch.
            if test_data:
                x_test, y_test = test_data
                accuracy = self.evaluate(x_test, y_test)
                tqdm.write(f"Epoch: {epoch}, Accuracy: {accuracy * 100:.2f} %")
            else:
                tqdm.write(f"Epoch: {epoch}")

In [ ]:
def assert_eq(actual: list, expected: list, msg: str = "") -> None:
    """Assert that two lists are equal."""
    if actual != expected:
        raise AssertionError(f"Mismatched {msg}\nGot: {actual}\nExpected: {expected}.")

## Exercise 1: Network (backprop_single)

Complete the implementation of `backprop_single` in `Network` below.
Your goal is to compute $\frac{d \mathcal{L}\left(\text{model}(x), y\right)}{d p}$ for each parameter $p$ of the network.


In [ ]:
class Network(BaseNetwork):
    def __init__(self, sizes: list[int]) -> None:
        super().__init__(sizes)
        # Recall that:
        # self.sizes = [N^0, ... , N^last]
        # self.biases is list of len(sizes) - 1 vectors of shapes (N^1), (N^2), ..., (N^last).
        # self.weights is a list of len(sizes) - 1 matrices of shape (N^{i+1}, N^i).

    def backprop_single(
        self, x: FloatNDArray, y: FloatNDArray
    ) -> tuple[list[FloatNDArray], list[FloatNDArray]]:
        """
        Backpropagation for a single input (not vectorized).

        Args:
        - x: input features, shape (N^0).
        - y: target label (one-hot encoded), shape (N^last).

        Returns (grads_w, grads_b), where:
        - grads_w is a list of gradients over weights for each layer, shapes (N^{i+1}, N^i).
        - grads_b is a list of gradients over biases for each layer, shapes (N^i).
        """
        # First, go forward, remembering all activations.
        # Activations (including inputs to the first layer), shapes (N^0), (N^1), ..., (N^last).
        gs: list[FloatNDArray] = [x]

        ### TODO ###


        assert_eq([g.shape for g in gs], [(n,) for n in self.sizes], "gs shapes")
        _loss = np.mean((gs[-1] - y) ** 2)

        # Now go backward from the final cost applying backpropagation.
        # Start with grad_g = gradient of MSE loss over output activations.
        # grad_g has shape initially (N^last), then go layer by layer.
        L = len(self.sizes) - 1
        N_last = self.sizes[L]
        grad_g = (2 / N_last) * (gs[L] - y.astype(np.float64))

        grads_w: list[FloatNDArray | None] = [None] * L
        grads_b: list[FloatNDArray | None] = [None] * L

        for i in range(L, 0, -1):  # i = L, L-1, ..., 1
            g = gs[i]  # shape (N^i)
            assert grad_g.shape == g.shape, f"{grad_g.shape=} vs {g.shape=}"

            ### TODO ###

        assert_eq(
            [grad_w.shape if grad_w is not None else None for grad_w in grads_w],
            [w.shape for w in self.weights],
            "grads_w shapes",
        )
        assert_eq(
            [grad_b.shape if grad_b is not None else None for grad_b in grads_b],
            [b.shape for b in self.biases],
            "grads_b shapes",
        )
        return cast(list[FloatNDArray], grads_w), cast(list[FloatNDArray], grads_b)

In [ ]:
# Smaller test on part of the dataset.
# The unvectorized version should take about 2s per epoch
# and achieve accuracy ~75% or more (for most executions).
np.random.seed(42)
trainer = Trainer(Network([784, 30, 10]))
trainer.train(
    (x_train[:3000], y_train[:3000]),
    test_data=(x_test[:3000], y_test[:3000]),
    epochs=5,
    mini_batch_size=100,
    learning_rate=10.0,
)

In [ ]:
%%time
# Full test.
# The unvectorized version should take about 20s per epoch
# and achieve accuracy ~90% or more.
np.random.seed(42)
trainer = Trainer(Network([784, 30, 10]))
trainer.train(
    (x_train, y_train),
    test_data=(x_test, y_test),
    epochs=10,
    mini_batch_size=100,
    learning_rate=5.0,
)

## Exercise 2: vectorized

Implement a "fully vectorized" version, i.e. one using matrix operations instead of going over examples one by one within a minibatch.


In [ ]:
class NetworkVectorized(BaseNetwork):
    """Multi-Layer Perceptron with vectorized (batched) methods."""

    def __init__(self, sizes: Sequence[int] = (784, 30, 10)) -> None:
        super().__init__(sizes)
        # Recall that:
        # self.sizes = [N^0, ... , N^last]
        # self.biases is list of len(sizes) - 1 vectors of shapes (N^1), (N^2), ..., (N^last).
        # self.weights is list of len(sizes) - 1 matrices of shapes (N^1, N^0), (N^2, N^1), ..., (N^last, N^{last-1}).

    @override
    def feedforward(self, x_mini_batch: FloatNDArray) -> FloatNDArray:
        """
        Run the network on a batch of cases of shape (B, N^0).
        Returns last layer activations, shape (B, N^last).
        """
        g = x_mini_batch
        ### TODO ###
        return g

    @override
    def backprop(
        self, x_mini_batch: FloatNDArray, y_mini_batch: FloatNDArray
    ) -> tuple[list[FloatNDArray], list[FloatNDArray]]:
        """Backpropagation for a mini-batch (vectorized).

        Args:
        - x_mini_batch: input, shape (B, N^0)
        - y_mini_batch: target label (one-hot encoded), shape (B, N^last)

        Returns (grads_w, grads_b), where:
        - grads_w: list of gradients over weights (shape (N^{i+1}, N^i)), for each layer.
        - grads_b: list of gradients over biases (shape (N^i)), for each layer.
        """
        B = len(x_mini_batch)

        # First, go forward, remembering all activations.
        # Activations (including inputs to the first layer), shapes (B, N^0), (B, N^1), ..., (B, N^last).
        gs: list[FloatNDArray] = [x_mini_batch]

        ### TODO ###


        assert_eq([g.shape for g in gs], [(B, n) for n in self.sizes], "gs shapes")

        # Now go backward from the final cost applying backpropagation.
        # Start with grad_g = gradient of MSE loss over output activations.
        # Remember that the mini-batch loss is the mean of the individual losses!
        # grad_g has shape initially (B, N^last), then go layer by layer.
        L = len(self.sizes) - 1
        N_last = self.sizes[L]

        grads_w: list[FloatNDArray | None] = [None] * L
        grads_b: list[FloatNDArray | None] = [None] * L

        ### TODO ###

        assert_eq(
            [grad_w.shape if grad_w is not None else None for grad_w in grads_w],
            [w.shape for w in self.weights],
            "grads_w shapes",
        )
        assert_eq(
            [grad_b.shape if grad_b is not None else None for grad_b in grads_b],
            [b.shape for b in self.biases],
            "grads_b shapes",
        )
        return cast(list[FloatNDArray], grads_w), cast(list[FloatNDArray], grads_b)

In [ ]:
%%time
# The vectorized version takes about ~1s per epoch.
np.random.seed(42)
trainer = Trainer(NetworkVectorized([784, 30, 10]))
trainer.train(
    (x_train, y_train),
    test_data=(x_test, y_test),
    epochs=50,
    mini_batch_size=100,
    learning_rate=5.0,
)

# Exercise 3 (optional)

The standard backpropagation method requires memorization of all outputs of all layers computed during the forward pass, for use in the backward pass, which can take a lot of precious GPU memory.
Instead of doing that, one can checkpoint (memorize) only activations of a select few layers and then recompute the rest as they are needed (redoing the forward pass between checkpoints).  
Your task is to complete the code below to implement backpropagation with checkpoints.

Question: Why didn't we need to store pre-activations in the above implementations?

In [ ]:
class NetworkWithCheckpoints(NetworkVectorized):
    """Multi-Layer Perceptron with gradient checkpointing."""

    def __init__(
        self, sizes: Sequence[int] = (784, 30, 10), checkpoints: Sequence[int] = (1,)
    ):
        """
        Args:
        - sizes: sequence of layer widths [N^0, ... , N^last]
          These are lengths of activation vectors, where:
          - N^0 is input size: 784.
          - N^last is the number of classes into which we can classify each input: 10.
        - checkpoints: Indices of layers whose activations we want to checkpoint,
                       between 1 and last (L = len(sizes) - 1).
        """
        super().__init__(sizes)
        L = len(sizes) - 1
        # Always checkpoint the input and last layer, remove duplicates.
        self.checkpoints = sorted(set(checkpoints) | {0, L})

    def backprop_generator(
        self, x: FloatNDArray, y: FloatNDArray
    ) -> Iterator[tuple[FloatNDArray, FloatNDArray]]:
        """
        Backpropagation for a mini-batch (vectorized).

        Args:
        - x: input, shape (B, N^0)
        - y: target label (one-hot encoded), shape (B, N^last)

        Yields (grad_w, grad_b) for each layer from i=last down to i=1, where:
        - grads_w: shape (N^i, N^{i-1})).
        - grads_b: shape (N^i)).
        """
        B = len(x)

        # Go forward, remembering only some activations and no pre-activations.

        # Values after activation function (including inputs to the first layer),
        # shapes (B, N^0), (B, N^1), ..., (B, N^last);
        # layers that are not checkpointed get None instead.
        gs: list[FloatNDArray | None] = [x]

        ### TODO ###

        # Now go backward from the final cost applying backpropagation.
        # Start with grad_g = gradient of MSE loss over output activations.
        # Remember that the mini-batch loss is the mean of the individual losses.
        # grad_g has shape initially (B, N^last), then go layer by layer.
        L = len(self.sizes) - 1
        N_last = self.sizes[L]

        # TODO
        g = gs[L]
        assert g is not None
        grad_g = (2 / (B * N_last)) * (g - y.astype(np.float64))


        # TODO backward pass.
        # For each interval between consecutive checkpoints:
        for start_i, end_i in ...:
            # Forward pass to restore intermediate activations.
            # for all layers in interval:
            # ...

            # Now g and grad_g have shape (B, N^{end_i}).
            assert g.shape == grad_g.shape == (B, self.sizes[end_i])

            # Backward pass between the checkpoints.
            # for all layers in the interval, backwards:
            #   # Compute grad_w, grad_b.
            #   # Yield them as we go, instead of returning a list.
            #   # https://docs.python.org/3/tutorial/classes.html#generators
            #   yield (grad_w, grad_b)

            # Now grad_g has shape (B, N^{start_i}).
            assert grad_g.shape == (B, self.sizes[start_i])


class TrainerWithCheckpoints(Trainer):
    @override
    def learning_step(
        self,
        x_mini_batch: FloatNDArray,
        y_mini_batch: FloatNDArray,
        learning_rate: float,
    ) -> None:
        """
        Update network parameters with a single mini-batch step of backpropagation and gradient descent.

        Args:
        - x_mini_batch: shape (B, N^0) where B is mini_batch_size.
        - y_mini_batch: shape (B, N^last).
        - learning_rate.
        """
        assert isinstance(self.network, NetworkWithCheckpoints)

        L = len(self.network.sizes) - 1
        indices = range(L, 0, -1)  # L, ..., 1

        for i, (grad_w, grad_b) in zip(
            indices,
            self.network.backprop_generator(x_mini_batch, y_mini_batch),
            strict=True,
        ):
            self.network.weights[i - 1] = (
                self.network.weights[i - 1] - learning_rate * grad_w
            )
            self.network.biases[i - 1] = (
                self.network.biases[i - 1] - learning_rate * grad_b
            )


In [ ]:
# %%time

# Compare on the same seed, results should be exactly the same.
np.random.seed(42)
trainer = TrainerWithCheckpoints(
    NetworkWithCheckpoints([784, 20, 15, 13, 10], checkpoints=[2])
)
trainer.train(
    (x_train, y_train),
    test_data=(x_test, y_test),
    epochs=1,
    mini_batch_size=100,
    learning_rate=50.0,
)

In [ ]:
# %%time

np.random.seed(42)
trainer = Trainer(NetworkVectorized([784, 20, 15, 13, 10]))
trainer.train(
    (x_train, y_train),
    test_data=(x_test, y_test),
    epochs=1,
    mini_batch_size=100,
    learning_rate=50.0,
)

 Memory usage can be checked by installing `memory_profiler` and using `%%memit` (instead of `%%time`).
 It will be hard to see a difference from checkpointing for an (fully connected) MLP.
 Consider what batch size, layer sizes and checkpoint placements could see the most benefit
 (don't worry about plummetting accuracy).

In [ ]:
%pip install memory_profiler
%load_ext memory_profiler

In [ ]:
%%memit

sizes = [784, 20, 15, 13, 10]
checkpoints = [2]
mini_batch_size = 100

np.random.seed(42)
trainer = TrainerWithCheckpoints(NetworkWithCheckpoints(sizes, checkpoints=checkpoints))
trainer.train(
    (x_train, y_train),
    test_data=(x_test, y_test),
    epochs=1,
    mini_batch_size=mini_batch_size,
    learning_rate=50.0,
)


In [ ]:
%%memit

np.random.seed(42)
trainer = Trainer(NetworkVectorized(sizes))
trainer.train(
    (x_train, y_train),
    test_data=(x_test, y_test),
    epochs=1,
    mini_batch_size=mini_batch_size,
    learning_rate=50.0,
)

# JAX Playground (Optional)
JAX is a framework that allows the creation of neural networks with numpy-like syntax.  
In this course, we will use Pytorch instead of JAX, but for this lab scenario, JAX can help us test our gradient computation implementation.  
Let's give it a try  

In [ ]:
%pip3 install jax

In [ ]:
import jax
import jax.numpy as jnp
from textwrap import dedent


def jax_sigmoid(z: jax.Array) -> jax.Array:
    return 1.0 / (1.0 + jnp.exp(-z))


def jax_sigmoid_prime(z: jax.Array) -> jax.Array:
    return jax_sigmoid(z) * (1 - jax_sigmoid(z))


# Define a jax function.
# We emphasize that this is a function, not a jax procedure,
# and in fact there are more requirements for writing good jax code,
# but this is just an example.
# (see https://jax.readthedocs.io/en/latest/tutorials.html)
def jax_forward(
    x: jax.Array, w: jax.Array, b: jax.Array
) -> tuple[jax.Array, jax.Array]:
    f = (w @ x.T).T + b  # shape (B, N1)
    g = jax_sigmoid(f)
    loss = g.sum()  # Just a dummy loss for simplicity.
    return loss, g


# This will calculate gradients for first, second, and third argument.
# has_aux tells that in addition to loss our function returns something more.
auto_backward = jax.value_and_grad(fun=jax_forward, argnums=[0, 1, 2], has_aux=True)


def manual_backward(
    x: jax.Array, w: jax.Array, b: jax.Array
) -> tuple[jax.Array, jax.Array, jax.Array]:
    ## TODO
    ###{
    f = (w @ x.T).T + b
    grad_g = jnp.ones_like(f)  # Grad of the dummy loss over activations g.
    ###}
    return grad_x, grad_w, grad_b


def example():
    B, N0, N1 = 3, 5, 7

    key = jax.random.key(42)
    key, subkey = jax.random.split(key)
    w = jax.random.normal(subkey, (N1, N0))
    key, subkey = jax.random.split(key)
    b = jax.random.normal(subkey, (N1,))
    x = jnp.arange(N0, dtype=w.dtype).reshape(1, N0) * jnp.ones((B, N0))

    (loss, res), (jax_dx, jax_dw, jax_db) = auto_backward(x, w, b)
    dx, dw, db = manual_backward(x, w, b)

    print(
        dedent(f"""
        diff dx = {jnp.mean(jnp.abs(jax_dx - dx))}
        diff dw = {jnp.mean(jnp.abs(jax_dw - dw))}
        diff db = {jnp.mean(jnp.abs(jax_db - db))}
        dtype={dx.dtype} (eps={jnp.finfo(dx.dtype).eps})
    """).strip()
    )


example()